# WoundScope — c7ec606 postprocessing recovery

這份 recovery notebook **不會重新訓練**。它固定接續已完成的 `c7ec6060f1bd` artifacts，先驗證 quick、full comparison、loss selection、multi-seed final 與 official validation 的檔案雜湊，再只恢復暫存資料、重跑 ONNX／benchmark／private gallery，最後建立 safe handoff ZIP。

> 若任何既有 training artifact 遺失或被修改，程式會立即停止，不會偷偷退回完整訓練。輸出是研究用 wound segmentation，不是診斷、嚴重度、預後或治療建議。

In [ ]:
#@title 1. Mount private Drive and locate the repaired source bundle
from google.colab import drive
from pathlib import Path
import os
runtime_root = Path(os.environ.setdefault('WOUNDSCOPE_RUNTIME_ROOT', str(Path.cwd()))).resolve()
os.chdir(runtime_root)
drive_mount = Path(os.environ.get('WOUNDSCOPE_DRIVE_MOUNT', str(runtime_root / 'drive')))
drive.mount(str(drive_mount))
drive_project_dir = drive_mount / 'MyDrive' / 'WoundScope'
source_zip = drive_project_dir / 'WoundScope_colab_source.zip'
artifact_base_dir = drive_project_dir / 'WoundScopeArtifacts'
if not source_zip.is_file():
    raise FileNotFoundError(f'Missing repaired source ZIP: {source_zip}')
print('Drive project:', drive_project_dir)
print('Repaired source ZIP:', source_zip)

In [ ]:
#@title 2. Verify the repair bundle and bind it to the completed c7ec606 run
import hashlib, json, re, shutil, zipfile
from pathlib import PurePosixPath
training_source_commit = 'c7ec6060f1bd0a813a890b95b50c2855d3c2640c'
with zipfile.ZipFile(source_zip) as archive:
    names = [item.filename for item in archive.infolist() if not item.is_dir()]
    for name in names:
        path = PurePosixPath(name)
        if not name or '\\' in name or path.is_absolute() or '..' in path.parts:
            raise RuntimeError(f'Unsafe source archive path: {name!r}')
    if len(names) != len(set(names)) or 'bundle_manifest.json' not in names:
        raise RuntimeError('Invalid or duplicate source bundle inventory')
    manifest = json.loads(archive.read('bundle_manifest.json'))
    if manifest.get('kind') != 'source' or manifest.get('schema_version') != 1:
        raise RuntimeError('Incompatible source bundle schema')
    expected_names = {'bundle_manifest.json'}
    for record in manifest['files']:
        content = archive.read(record['path'])
        expected_names.add(record['path'])
        if len(content) != record['size'] or hashlib.sha256(content).hexdigest() != record['sha256']:
            raise RuntimeError(f'Source bundle checksum mismatch: {record["path"]}')
    if set(names) != expected_names:
        raise RuntimeError('Source ZIP contains unlisted members')
    implementation_source_commit = manifest['source_commit']
    if not isinstance(implementation_source_commit, str) or re.fullmatch(r'[0-9a-f]{40}', implementation_source_commit) is None:
        raise RuntimeError(f'Invalid implementation source commit: {implementation_source_commit!r}')
    if implementation_source_commit == training_source_commit:
        raise RuntimeError('The uploaded ZIP is still the old c7ec606 bundle; replace it with the repaired ZIP')
    project_dir = runtime_root / f'WoundScope_repair_{implementation_source_commit[:12]}'
    if project_dir.exists():
        shutil.rmtree(project_dir)
    project_dir.mkdir(parents=True)
    for name in sorted(expected_names):
        target = project_dir.joinpath(*PurePosixPath(name).parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))
artifact_dir = artifact_base_dir / training_source_commit[:12]
state_path = artifact_dir / 'pipeline_state.json'
if not state_path.is_file():
    raise FileNotFoundError(f'Missing completed c7ec606 pipeline state: {state_path}')
print('Training source:', training_source_commit)
print('Repair implementation:', implementation_source_commit)
print('Reusing private artifacts:', artifact_dir)

In [ ]:
#@title 3. Install the repair source and enforce the CUDA gate
import platform, subprocess, sys
os.chdir(project_dir)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train,export,app]'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. Select an A100 GPU runtime; CPU fallback is forbidden.')
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
#@title 4. Verify completed training and resume postprocessing only
data_dir = Path(os.environ.get('WOUNDSCOPE_DATA_DIR', str(runtime_root / 'woundscope_data')))
command = [
    sys.executable, 'scripts/resume_colab_postprocessing.py',
    '--project-root', str(project_dir),
    '--data-dir', str(data_dir),
    '--artifact-dir', str(artifact_dir),
    '--source-commit', training_source_commit,
    '--implementation-source-commit', implementation_source_commit,
]
before_stage_records = {}
if state_path.is_file():
    before_stage_records = json.loads(state_path.read_text(encoding='utf-8')).get('stages', {})
completed = subprocess.run(command, check=False)
if completed.returncode != 0:
    if state_path.is_file():
        state = json.loads(state_path.read_text(encoding='utf-8'))
        failures = [(stage, record) for stage, record in state.get('stages', {}).items() if record.get('status') == 'failed' and record.get('implementation_source_commit') == implementation_source_commit and record != before_stage_records.get(stage)]
        if failures:
            stage, record = failures[-1]
            raise RuntimeError(f'Postprocessing failed: {stage} — {record.get("error_type")}: {record.get("error")}')
    raise RuntimeError(f'Postprocessing recovery exited with code {completed.returncode}')
handoff = artifact_dir / 'handoff' / f'woundscope_colab_results_{training_source_commit[:12]}.zip'
if not handoff.is_file():
    raise RuntimeError('Recovery completed without the required safe handoff ZIP')
print('SAFE_HANDOFF_READY:', handoff)
print('Training was not rerun.')